In [1]:
%pip install tensorflow pillow matplotlib pandas numpy scikit-learn pytesseract opencv-python-headless ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [1]:
from pathlib import Path
import json
import tensorflow as tf
import matplotlib.pyplot as plt

DATASET_DIR = Path(r'dataset')
OUTPUT_DIR = Path('../ml_models')
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 16
SEED = 42
EPOCHS_HEAD = 2
EPOCHS_FINE_TUNE = 2

assert DATASET_DIR.exists(), f'Dataset folder not found: {DATASET_DIR.resolve()}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR, validation_split=0.2, subset='training', seed=SEED,
    image_size=IMAGE_SIZE, batch_size=BATCH_SIZE
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR, validation_split=0.2, subset='validation', seed=SEED,
    image_size=IMAGE_SIZE, batch_size=BATCH_SIZE
)
class_names = train_ds.class_names
print('Classes:', class_names)
(OUTPUT_DIR / 'class_names.json').write_text(
    json.dumps(class_names, indent=2), encoding='utf-8'
)


Found 60947 files belonging to 38 classes.
Using 48758 files for training.
Found 60947 files belonging to 38 classes.
Using 12189 files for validation.
Classes: ['Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy', 'Onion_Bulb_blight_D', 'Onion__Alternaria_D', 'Onion__Botrytis_Leaf_Blight', 'Onion__Bulb_Rot', 'Onion__Caterpillar_P', 'Onion__Downy_mildew', 'Onion__Fusarium_D', 'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew', 'Strawberry___Leaf_scorch', 'Strawberry___healthy', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late

1146

In [3]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal_and_vertical'),
    tf.keras.layers.RandomRotation(0.12),
    tf.keras.layers.RandomZoom(0.12),
    tf.keras.layers.RandomContrast(0.12),
], name='augmentation')


In [4]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMAGE_SIZE + (3,), include_top=False, weights='imagenet'
)
base_model.trainable = False

inputs = tf.keras.Input(shape=IMAGE_SIZE + (3,))
x = augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.25)(x)
outputs = tf.keras.layers.Dense(len(class_names), activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ augmentation (Sequential)       │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 38)             │        48,678 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,306,662 (8.80 MB)

 Trainable params: 48,678 (190.15 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.3),
]
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD, callbacks=callbacks)


Epoch 1/2


C:\Users\Brightech\anaconda3\envs\plant_disease\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


In [6]:
base_model.trainable = True
for layer in base_model.layers[:-35]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
fine_history = model.fit(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS_FINE_TUNE, callbacks=callbacks
)


Epoch 1/2
3048/3048 ━━━━━━━━━━━━━━━━━━━━ 2368s 704ms/step - accuracy: 0.8590 - loss: 0.4611 - val_accuracy: 0.9231 - val_loss: 0.2348 - learning_rate: 1.0000e-05
Epoch 2/2
3048/3048 ━━━━━━━━━━━━━━━━━━━━ 2168s 658ms/step - accuracy: 0.9169 - loss: 0.2457 - val_accuracy: 0.9392 - val_loss: 0.1898 - learning_rate: 1.0000e-05


In [8]:
val_loss, val_accuracy = model.evaluate(val_ds)
print(f'Validation accuracy: {val_accuracy:.4f}')
model.save(OUTPUT_DIR / 'plant_disease_model.keras')
print('Saved model and class names to:', OUTPUT_DIR.resolve())


NameError: name 'model' is not defined

In [5]:
# Set Test Images Folder
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# Change this path when necessary
TEST_DIR = Path(
    r"C:\Users\Brightech\Downloads\test_images"
)

SUPPORTED_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
    ".tif",
    ".tiff"
}

assert TEST_DIR.exists(), (
    f"Test-images folder not found:\n{TEST_DIR}"
)

test_image_paths = sorted(
    image_path
    for image_path in TEST_DIR.rglob("*")
    if image_path.is_file()
    and image_path.suffix.lower() in SUPPORTED_EXTENSIONS
)

print("Total test images found:", len(test_image_paths))

if len(test_image_paths) == 0:
    print("No supported images found inside:", TEST_DIR)

Total test images found: 3


In [6]:
# Prediction Function
def predict_test_image(image_path):

    # Open original image
    original_image = Image.open(
        image_path
    ).convert("RGB")

    # Resize exactly as required by MobileNetV2
    resized_image = original_image.resize(
        IMAGE_SIZE,
        Image.Resampling.LANCZOS
    )

    # Convert image to NumPy array
    image_array = np.asarray(
        resized_image,
        dtype=np.float32
    )

    # Add batch dimension:
    # (224, 224, 3) becomes (1, 224, 224, 3)
    image_batch = np.expand_dims(
        image_array,
        axis=0
    )

    # Do not manually apply preprocess_input here.
    # Your trained model already contains preprocessing.
    probabilities = model.predict(
        image_batch,
        verbose=0
    )[0]

    predicted_index = int(
        np.argmax(probabilities)
    )

    predicted_disease = class_names[
        predicted_index
    ]

    confidence_percent = float(
        probabilities[predicted_index] * 100
    )

    result = {
        "file_name": image_path.name,
        "predicted_disease": predicted_disease,
        "confidence_percent": round(
            confidence_percent,
            2
        ),
        "status": "Success",
        "image_name": image_path.name,
        "image_path": str(image_path)
    }

    return result, original_image, probabilities

In [7]:
# Test all present Images
all_test_results = []

print("Total test images found:", len(test_image_paths))

for image_number, image_path in enumerate(
    test_image_paths,
    start=1
):

    print()
    print("=" * 75)

    print(
        f"Testing image "
        f"{image_number}/{len(test_image_paths)}"
    )

    print("Image:", image_path)

    print("=" * 75)

    try:

        result, original_image, probabilities = (
            predict_test_image(image_path)
        )

        all_test_results.append(result)

        # Display result table
        current_result_df = pd.DataFrame(
            [result]
        )

        display(current_result_df)

        # Display image with prediction
        plt.figure(figsize=(8, 7))

        plt.imshow(original_image)

        plt.title(
            f"{result['file_name']}\n"
            f"{result['predicted_disease']} "
            f"({result['confidence_percent']:.2f}%)"
        )

        plt.axis("off")
        plt.tight_layout()
        plt.show()

    except Exception as error:

        error_result = {
            "file_name": image_path.name,
            "predicted_disease": "",
            "confidence_percent": np.nan,
            "status": f"Error: {error}",
            "image_name": image_path.name,
            "image_path": str(image_path)
        }

        all_test_results.append(
            error_result
        )

        display(
            pd.DataFrame([error_result])
        )

Total test images found: 3

Testing image 1/3
Image: C:\Users\Brightech\Downloads\test_images\sample1.jpg


,file_name,predicted_disease,confidence_percent,status,image_name,image_path
0,sample1.jpg,,NaN,Error: name 'IMAGE_SIZE' is not defined,sample1.jpg,C:\Users\Brightech\Downloads\test_images\sampl...



Testing image 2/3
Image: C:\Users\Brightech\Downloads\test_images\sample2.jpg


,file_name,predicted_disease,confidence_percent,status,image_name,image_path
0,sample2.jpg,,NaN,Error: name 'IMAGE_SIZE' is not defined,sample2.jpg,C:\Users\Brightech\Downloads\test_images\sampl...



Testing image 3/3
Image: C:\Users\Brightech\Downloads\test_images\sample3.JPG


,file_name,predicted_disease,confidence_percent,status,image_name,image_path
0,sample3.JPG,,NaN,Error: name 'IMAGE_SIZE' is not defined,sample3.JPG,C:\Users\Brightech\Downloads\test_images\sampl...


In [ ]:
# Final Summary & Saving
all_test_results_df = pd.DataFrame(
    all_test_results
)

print()
print("=" * 75)
print("ALL TEST IMAGES COMPLETED")
print("=" * 75)

display(all_test_results_df)

successful_predictions = int(
    all_test_results_df[
        "status"
    ].eq("Success").sum()
)

failed_predictions = int(
    len(all_test_results_df)
    - successful_predictions
)

print(
    "Total test images:",
    len(all_test_results_df)
)

print(
    "Successful predictions:",
    successful_predictions
)

print(
    "Failed predictions:",
    failed_predictions
)

csv_output_path = (
    OUTPUT_DIR
    / "test_image_predictions.csv"
)

all_test_results_df.to_csv(
    csv_output_path,
    index=False
)

print(
    "Results CSV saved at:",
    csv_output_path
)